# Multi-Coin LSTM Training Pipeline with MLflow Integration

This notebook implements a complete training pipeline for binary classification of trading signals across multiple cryptocurrencies.

## Pipeline Overview:
1. **Data Loading**: Load pre-optimized CSV files from vectorbt_optimizer output
2. **Preprocessing**: Use VectorBTDataPreprocessor for column filtering, scaling, and sequence creation
3. **Multi-Coin Concatenation**: Combine all coins into unified train/val/test datasets
4. **Model Training**: Train CNNLSTMSignalPredictor with MLflow tracking
5. **Evaluation**: Comprehensive metrics and visualizations

## Section 1: Setup and Configuration

In [1]:
# Cell 1: Imports and Path Setup
import sys
from pathlib import Path
import json
import os
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict, field
import pickle

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# LSTM modules
from crypto_analysis.lstm.model import ModelConfig, CNNLSTMSignalPredictor
from crypto_analysis.lstm.trainer import Trainer, TrainingConfig, TrainingHistory
from crypto_analysis.lstm.loss import BinarySignalLoss, FocalBinaryLoss
from crypto_analysis.lstm.dataset import SignalDataset, create_sequences

# VectorBT Data Preprocessor (replaces DataPreprocessor and DatasetBuilder)
from crypto_analysis.vectorbt_optimizer import VectorBTDataPreprocessor

# MLflow
import mlflow
import mlflow.pytorch

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"MLflow version: {mlflow.__version__}")

PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4070 Ti
MLflow version: 3.8.1


In [2]:
# Cell 2: Configuration Constants

# === CONFIGURATION ===

# Data paths - VectorBT optimized CSV files
CSV_DIR = Path("notebooks/csvs")  # Directory with vectorbt_optimizer output CSVs
OUTPUT_DIR = Path("notebooks/coin_csvs")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)

# VectorBTDataPreprocessor configuration
PREPROCESSOR_CONFIG = {
    "remove_raw_indicators": False,   # Keep only OHLCV + *_entry/*_exit signals
    "target_shift": 1,               # Features at t predict target at t+1
    "sequence_length": 36,           # LSTM input sequence length
    "stride": 1,                     # Step between sequences
    "scaler_type": "standard",       # 'standard' or 'minmax'
    "target_column": "tradeable",
}

# Train/val/test split ratios
SPLIT_CONFIG = {
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "test_ratio": 0.2,
}

# MLflow experiment name
MLFLOW_EXPERIMENT = "multi_coin_lstm_training"

print("Configuration loaded.")
print(f"CSV directory: {CSV_DIR.absolute()}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")
print(f"Sequence length: {PREPROCESSOR_CONFIG['sequence_length']}")
print(f"Target shift: {PREPROCESSOR_CONFIG['target_shift']}")
print(f"Remove raw indicators: {PREPROCESSOR_CONFIG['remove_raw_indicators']}")

Configuration loaded.
CSV directory: /workspace/trade-automation/notebooks/csvs
Output directory: /workspace/trade-automation/notebooks/coin_csvs
Sequence length: 36
Target shift: 1
Remove raw indicators: False


In [3]:
# Cell 3: Load Data from CSV Directory

def load_vectorbt_csvs(
    csv_dir: Path,
    pattern: str = "*_optimized.csv",
    verbose: bool = True
) -> Dict[str, pd.DataFrame]:
    """
    Load all vectorbt optimizer output CSVs from directory.
    
    Parameters
    ----------
    csv_dir : Path
        Directory containing CSV files
    pattern : str
        Glob pattern for CSV files (default: *_optimized.csv)
    verbose : bool
        Print progress
    
    Returns
    -------
    Dict[str, pd.DataFrame]
        Symbol -> DataFrame mapping
    """
    if verbose:
        print(f"Loading CSVs from: {csv_dir.absolute()}")
        print(f"Pattern: {pattern}")
        print("-" * 60)
    
    # Use VectorBTDataPreprocessor's static method
    dfs = VectorBTDataPreprocessor.load_directory(csv_dir, pattern)
    
    if not dfs:
        print(f"Warning: No CSV files found matching pattern '{pattern}' in {csv_dir}")
        return {}
    
    if verbose:
        for name, df in dfs.items():
            trade_count = (df['tradeable'] == 'trade').sum() if 'tradeable' in df.columns else 0
            hold_count = (df['tradeable'] == 'hold').sum() if 'tradeable' in df.columns else 0
            total = len(df)
            trade_pct = trade_count / total * 100 if total > 0 else 0
            print(f"  {name}: {df.shape} | Trade: {trade_count} ({trade_pct:.1f}%)")
        print("-" * 60)
        print(f"Loaded {len(dfs)} dataframes")
    
    return dfs


def print_data_summary(dataframes: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Print summary statistics for all coin dataframes."""
    print("=" * 80)
    print("DATA SUMMARY")
    print("=" * 80)
    
    summary_data = []
    metadata_cols = ['date', 'tradeable']
    
    for name, df in dataframes.items():
        n_rows = len(df)
        n_cols = len(df.columns)
        
        # Count entry/exit signal columns
        entry_cols = [c for c in df.columns if c.endswith('_entry')]
        exit_cols = [c for c in df.columns if c.endswith('_exit')]
        
        trade_count = (df['tradeable'] == 'trade').sum() if 'tradeable' in df.columns else 0
        hold_count = (df['tradeable'] == 'hold').sum() if 'tradeable' in df.columns else 0
        trade_pct = trade_count / n_rows * 100 if n_rows > 0 else 0
        
        summary_data.append({
            'Name': name,
            'Rows': n_rows,
            'Columns': n_cols,
            'Entry Signals': len(entry_cols),
            'Exit Signals': len(exit_cols),
            'Trade': trade_count,
            'Hold': hold_count,
            'Trade %': f"{trade_pct:.2f}%"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    print("=" * 80)
    return summary_df


# Load CSVs from directory
coin_dataframes = load_vectorbt_csvs(CSV_DIR, pattern="*_optimized.csv", verbose=True)

# Print data summary
if coin_dataframes:
    data_summary = print_data_summary(coin_dataframes)
    symbols = list(coin_dataframes.keys())
    print(f"\nLoaded symbols ({len(symbols)}): {symbols}")
else:
    print("\nNo data loaded. Please ensure CSV files exist in the specified directory.")
    print(f"Expected location: {CSV_DIR.absolute()}")
    print("Run vectorbt_optimizer.optimize_all() first to generate CSV files.")

Loading CSVs from: /workspace/trade-automation/notebooks/csvs
Pattern: *_optimized.csv
------------------------------------------------------------
  ADA_optimized: (9202, 198) | Trade: 5505 (59.8%)
  ALGO_optimized: (9202, 198) | Trade: 5991 (65.1%)
  ARB_optimized: (9202, 198) | Trade: 6271 (68.1%)
  ATOM_optimized: (9202, 198) | Trade: 4920 (53.5%)
  AVAX_optimized: (9202, 198) | Trade: 5637 (61.3%)
  BNB_optimized: (9202, 198) | Trade: 2361 (25.7%)
  BTC_optimized: (9202, 198) | Trade: 1700 (18.5%)
  CHZ_optimized: (9202, 198) | Trade: 5206 (56.6%)
  DOGE_optimized: (9202, 198) | Trade: 5388 (58.6%)
  DOT_optimized: (9202, 198) | Trade: 5328 (57.9%)
  ETH_optimized: (9202, 198) | Trade: 3785 (41.1%)
  FIL_optimized: (9202, 198) | Trade: 5414 (58.8%)
  HBAR_optimized: (9202, 198) | Trade: 5839 (63.5%)
  ICP_optimized: (9202, 198) | Trade: 5770 (62.7%)
  IOTA_optimized: (9202, 198) | Trade: 6278 (68.2%)
  LDO_optimized: (9202, 198) | Trade: 6655 (72.3%)
  LINK_optimized: (9202, 198) 

## Section 2: Initialize Preprocessor and Create Sequences

In [4]:
# Cell 4: Initialize VectorBTDataPreprocessor

# Create preprocessor with configuration
preprocessor = VectorBTDataPreprocessor(
    remove_raw_indicators=PREPROCESSOR_CONFIG["remove_raw_indicators"],
    target_shift=PREPROCESSOR_CONFIG["target_shift"],
    sequence_length=PREPROCESSOR_CONFIG["sequence_length"],
    stride=PREPROCESSOR_CONFIG["stride"],
    scaler_type=PREPROCESSOR_CONFIG["scaler_type"],
    target_column=PREPROCESSOR_CONFIG["target_column"],
)

print("VectorBTDataPreprocessor initialized:")
print(preprocessor)

# Fit preprocessor on all data
if coin_dataframes:
    preprocessor.fit(coin_dataframes)
    print(f"\nAfter fitting:")
    print(preprocessor)
    print(f"\nFeature columns: {preprocessor.get_feature_names()}")
    print(f"Number of features: {preprocessor.get_num_features()}")
    
    # Store for later use
    n_features = preprocessor.get_num_features()
    feature_names = preprocessor.get_feature_names()
else:
    print("No data to fit preprocessor on.")

VectorBTDataPreprocessor initialized:
VectorBTDataPreprocessor(sequence_length=36, target_shift=1, stride=1, remove_raw_indicators=False, scaler_type='standard', status=unfitted)

After fitting:
VectorBTDataPreprocessor(sequence_length=36, target_shift=1, stride=1, remove_raw_indicators=False, n_features=177, n_ohlcv=5, n_signals=172, scaler_type='standard')

Feature columns: ['open', 'high', 'low', 'close', 'volume', 'RSI_entry', 'RSI_exit', 'RSI_rsi', 'STOCH_entry', 'STOCH_exit', 'STOCH_slowk', 'STOCH_slowd', 'STOCHRSI_entry', 'STOCHRSI_exit', 'STOCHRSI_fastk', 'STOCHRSI_fastd', 'CCI_entry', 'CCI_exit', 'CCI_cci', 'MFI_entry', 'MFI_exit', 'MFI_mfi', 'WILLR_entry', 'WILLR_exit', 'WILLR_willr', 'CMO_entry', 'CMO_exit', 'CMO_cmo', 'ADX_entry', 'ADX_exit', 'ADX_adx', 'MOM_entry', 'MOM_exit', 'MOM_mom', 'ROC_entry', 'ROC_exit', 'ROC_roc', 'TRIX_entry', 'TRIX_exit', 'TRIX_trix', 'ULTOSC_entry', 'ULTOSC_exit', 'ULTOSC_ultosc', 'BOP_entry', 'BOP_exit', 'BOP_bop', 'AROON_entry', 'AROON_exit',

In [5]:
# Cell 5: Create Sequences for Each Coin (Separately for Proper Splitting)

def create_coin_sequences_separate(
    preprocessor: VectorBTDataPreprocessor,
    dataframes: Dict[str, pd.DataFrame],
    verbose: bool = True
) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """
    Create sequences for each coin separately.
    
    This allows us to split each coin's data before concatenating,
    ensuring no data leakage between train/val/test sets.
    
    Parameters
    ----------
    preprocessor : VectorBTDataPreprocessor
        Fitted preprocessor
    dataframes : Dict[str, pd.DataFrame]
        Symbol -> DataFrame mapping
    verbose : bool
        Print progress
    
    Returns
    -------
    Dict[str, Tuple[np.ndarray, np.ndarray]]
        Symbol -> (X sequences, y sequences) mapping
    """
    sequences = {}
    
    if verbose:
        print("Creating sequences for each coin...")
        print(f"  Sequence length: {preprocessor.sequence_length}")
        print(f"  Target shift: {preprocessor.target_shift}")
        print(f"  Stride: {preprocessor.stride}")
        print("-" * 60)
    
    for name, df in dataframes.items():
        try:
            # Transform to features/targets
            features, targets = preprocessor.transform(df)
            
            # Create sequences
            X, y = preprocessor.create_sequences(features, targets)
            sequences[name] = (X, y)
            
            if verbose:
                trade_pct = (y == 1).mean() * 100
                print(f"  {name}: X={X.shape}, y={y.shape} | Trade: {trade_pct:.1f}%")
                
        except Exception as e:
            print(f"  {name}: Error - {e}")
    
    if verbose:
        print("-" * 60)
        total_seqs = sum(X.shape[0] for X, y in sequences.values())
        print(f"Total sequences: {total_seqs}")
    
    return sequences


# Create sequences for each coin
if coin_dataframes:
    coin_sequences = create_coin_sequences_separate(preprocessor, coin_dataframes, verbose=True)
else:
    coin_sequences = {}
    print("No dataframes to create sequences from.")

Creating sequences for each coin...
  Sequence length: 36
  Target shift: 1
  Stride: 1
------------------------------------------------------------
  ADA_optimized: X=(8961, 36, 177), y=(8961,) | Trade: 59.4%
  ALGO_optimized: Error - Missing feature columns: {'MACDEXT_macdsignal', 'MACDEXT_macd', 'MACDEXT_macdhist', 'MACDEXT_entry', 'MACDEXT_exit'}
  ARB_optimized: Error - Missing feature columns: {'MACDEXT_macdsignal', 'CCI_entry', 'MACDEXT_macd', 'CCI_exit', 'MACDEXT_macdhist', 'SMA_entry', 'MACDEXT_entry', 'SMA_exit', 'MACDEXT_exit'}
  ATOM_optimized: X=(8933, 36, 177), y=(8933,) | Trade: 53.4%
  AVAX_optimized: Error - Missing feature columns: {'SMA_entry', 'SMA_exit'}
  BNB_optimized: X=(8823, 36, 177), y=(8823,) | Trade: 26.0%
  BTC_optimized: Error - Missing feature columns: {'HT_DCPHASE_entry'}
  CHZ_optimized: Error - Missing feature columns: {'CMO_exit', 'MACDEXT_macdsignal', 'MACDEXT_macd', 'MACDEXT_macdhist', 'MACDEXT_entry', 'HT_DCPHASE_entry', 'MACDEXT_exit'}
  DOGE_opt

In [6]:
# Cell 6: Validate Target Sequences

def validate_target_sequences(
    sequences: Dict[str, Tuple[np.ndarray, np.ndarray]]
) -> Dict[str, Dict]:
    """
    Validate that target sequences have correct binary labels.
    
    Parameters
    ----------
    sequences : Dict[str, Tuple[np.ndarray, np.ndarray]]
        Symbol -> (X, y) mapping
    
    Returns
    -------
    Dict[str, Dict]
        Validation results per symbol
    """
    print("\n" + "=" * 70)
    print("TARGET SEQUENCE VALIDATION")
    print("=" * 70)
    
    results = {}
    
    for name, (X, y) in sequences.items():
        unique_targets = np.unique(y)
        hold_count = int((y == 0).sum())
        trade_count = int((y == 1).sum())
        total = len(y)
        
        # Check for valid binary labels
        is_valid = set(unique_targets).issubset({0, 1})
        
        results[name] = {
            "total_sequences": total,
            "hold_count": hold_count,
            "trade_count": trade_count,
            "hold_pct": hold_count / total * 100 if total > 0 else 0,
            "trade_pct": trade_count / total * 100 if total > 0 else 0,
            "unique_labels": unique_targets.tolist(),
            "valid": is_valid
        }
        
        status = "OK" if is_valid else "INVALID"
        print(f"{name}: {total:5d} seqs | Hold: {hold_count:5d} ({results[name]['hold_pct']:5.1f}%) | "
              f"Trade: {trade_count:5d} ({results[name]['trade_pct']:5.1f}%) | [{status}]")
    
    print("=" * 70)
    
    # Summary
    all_valid = all(r["valid"] for r in results.values())
    if all_valid:
        print("All sequences validated successfully - binary labels only.")
    else:
        invalid = [s for s, r in results.items() if not r["valid"]]
        print(f"WARNING: Invalid sequences found for: {invalid}")
    
    return results


# Validate sequences
if coin_sequences:
    sequence_validation = validate_target_sequences(coin_sequences)
else:
    sequence_validation = {}


TARGET SEQUENCE VALIDATION
ADA_optimized:  8961 seqs | Hold:  3640 ( 40.6%) | Trade:  5321 ( 59.4%) | [OK]
ATOM_optimized:  8933 seqs | Hold:  4166 ( 46.6%) | Trade:  4767 ( 53.4%) | [OK]
BNB_optimized:  8823 seqs | Hold:  6533 ( 74.0%) | Trade:  2290 ( 26.0%) | [OK]
DOGE_optimized:  8871 seqs | Hold:  3696 ( 41.7%) | Trade:  5175 ( 58.3%) | [OK]
DOT_optimized:  8571 seqs | Hold:  3658 ( 42.7%) | Trade:  4913 ( 57.3%) | [OK]
ETH_optimized:  8934 seqs | Hold:  5228 ( 58.5%) | Trade:  3706 ( 41.5%) | [OK]
ICP_optimized:  8904 seqs | Hold:  3345 ( 37.6%) | Trade:  5559 ( 62.4%) | [OK]
LDO_optimized:  8631 seqs | Hold:  2455 ( 28.4%) | Trade:  6176 ( 71.6%) | [OK]
LINK_optimized:  8589 seqs | Hold:  3507 ( 40.8%) | Trade:  5082 ( 59.2%) | [OK]
LTC_optimized:  8982 seqs | Hold:  4510 ( 50.2%) | Trade:  4472 ( 49.8%) | [OK]
SOL_optimized:  8880 seqs | Hold:  4033 ( 45.4%) | Trade:  4847 ( 54.6%) | [OK]
XRP_optimized:  8595 seqs | Hold:  4632 ( 53.9%) | Trade:  3963 ( 46.1%) | [OK]
All seque

In [7]:
## Section 3: Dataset Splitting and Concatenation

In [23]:
# Cell 7: Dataset Splitting and Concatenation Utilities

def split_coin_sequences(
    sequences: Dict[str, Tuple[np.ndarray, np.ndarray]],
    split_config: Dict
) -> Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]]:
    """
    Split each coin's sequences into train/val/test sets.
    Uses sequential split (no shuffle) to preserve temporal order.
    
    Parameters
    ----------
    sequences : Dict[str, Tuple[np.ndarray, np.ndarray]]
        Symbol -> (X, y) mapping
    split_config : Dict
        Split ratios (train_ratio, val_ratio, test_ratio)
    
    Returns
    -------
    Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]]
        Symbol -> {'train': (X, y), 'val': (X, y), 'test': (X, y)}
    """
    split_data = {}
    
    train_ratio = split_config["train_ratio"]
    val_ratio = split_config["val_ratio"]
    
    print(f"Splitting sequences: {train_ratio*100:.0f}% train, {val_ratio*100:.0f}% val, "
          f"{split_config['test_ratio']*100:.0f}% test")
    print("-" * 70)
    
    for name, (X, y) in sequences.items():
        n = len(X)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))
        
        split_data[name] = {
            'train': (X[:train_end], y[:train_end]),
            'val': (X[train_end:val_end], y[train_end:val_end]),
            'test': (X[val_end:], y[val_end:])
        }
        
        print(f"{name}: train={train_end}, val={val_end - train_end}, test={n - val_end}")
    
    return split_data


def concatenate_coin_datasets(
    split_data: Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]],
    device: Optional[torch.device] = None
) -> Dict[str, SignalDataset]:
    """
    Concatenate all coins' train/val/test sets into unified datasets.
    
    Parameters
    ----------
    split_data : Dict
        Split data from split_coin_sequences
    device : torch.device, optional
        Device for tensors
    
    Returns
    -------
    Dict[str, SignalDataset]
        {'train': SignalDataset, 'val': SignalDataset, 'test': SignalDataset}
    """
    combined = {
        'train': {'X': [], 'y': []},
        'val': {'X': [], 'y': []},
        'test': {'X': [], 'y': []}
    }
    
    for name, splits in split_data.items():
        for split_name in ['train', 'val', 'test']:
            X, y = splits[split_name]
            combined[split_name]['X'].append(X)
            combined[split_name]['y'].append(y)
    
    datasets = {}
    print("\n" + "=" * 50)
    print("CONCATENATED DATASETS")
    print("=" * 50)
    
    for split_name in ['train', 'val', 'test']:
        X_concat = np.concatenate(combined[split_name]['X'], axis=0)
        y_concat = np.concatenate(combined[split_name]['y'], axis=0)
        datasets[split_name] = SignalDataset(X_concat, y_concat, device=device)
        
        dist = datasets[split_name].get_class_distribution()
        print(f"{split_name:5s}: {len(datasets[split_name]):6d} samples | "
              f"Hold: {dist['hold']:5d} | Trade: {dist['trade']:5d}")
    
    print("=" * 50)
    return datasets


def analyze_dataset_distribution(
    datasets: Dict[str, SignalDataset]
) -> Dict[str, Dict]:
    """
    Analyze class distribution in each dataset split.
    
    Parameters
    ----------
    datasets : Dict[str, SignalDataset]
        Train/val/test datasets
    
    Returns
    -------
    Dict[str, Dict]
        Distribution statistics for each split
    """
    distributions = {}
    
    print("\n" + "=" * 70)
    print("DATASET DISTRIBUTION ANALYSIS")
    print("=" * 70)
    
    for split_name, dataset in datasets.items():
        dist = dataset.get_class_distribution()
        total = dist['hold'] + dist['trade']
        hold_pct = dist['hold'] / total * 100 if total > 0 else 0
        trade_pct = dist['trade'] / total * 100 if total > 0 else 0
        imbalance_ratio = dist['hold'] / max(dist['trade'], 1)
        
        distributions[split_name] = {
            'total': total,
            'hold': dist['hold'],
            'trade': dist['trade'],
            'hold_pct': hold_pct,
            'trade_pct': trade_pct,
            'imbalance_ratio': imbalance_ratio
        }
        
        print(f"\n{split_name.upper()}:")
        print(f"  Total samples: {total}")
        print(f"  Hold:  {dist['hold']:6d} ({hold_pct:5.2f}%)")
        print(f"  Trade: {dist['trade']:6d} ({trade_pct:5.2f}%)")
        print(f"  Imbalance ratio (hold/trade): {imbalance_ratio:.2f}")
    
    print("\n" + "=" * 70)
    return distributions

In [25]:
# Cell 8: Split and Concatenate Datasets

if coin_sequences:
    # Split each coin's sequences
    split_coin_data = split_coin_sequences(coin_sequences, SPLIT_CONFIG)
    
    # Concatenate into unified datasets
    multi_coin_datasets = concatenate_coin_datasets(split_coin_data, 'cuda')
    
    # Analyze distribution
    dataset_distributions = analyze_dataset_distribution(multi_coin_datasets)
else:
    print("No sequences to split.")
    split_coin_data = {}
    multi_coin_datasets = {}
    dataset_distributions = {}

Splitting sequences: 60% train, 20% val, 20% test
----------------------------------------------------------------------
ADA_optimized: train=5376, val=1792, test=1793
ATOM_optimized: train=5359, val=1787, test=1787
BNB_optimized: train=5293, val=1765, test=1765
DOGE_optimized: train=5322, val=1774, test=1775
DOT_optimized: train=5142, val=1714, test=1715
ETH_optimized: train=5360, val=1787, test=1787
ICP_optimized: train=5342, val=1781, test=1781
LDO_optimized: train=5178, val=1726, test=1727
LINK_optimized: train=5153, val=1718, test=1718
LTC_optimized: train=5389, val=1796, test=1797
SOL_optimized: train=5328, val=1776, test=1776
XRP_optimized: train=5157, val=1719, test=1719

CONCATENATED DATASETS
train:  63399 samples | Hold: 26973 | Trade: 36426
val  :  21135 samples | Hold: 11104 | Trade: 10031
test :  21140 samples | Hold: 11326 | Trade:  9814

DATASET DISTRIBUTION ANALYSIS

TRAIN:
  Total samples: 63399
  Hold:   26973 (42.54%)
  Trade:  36426 (57.46%)
  Imbalance ratio (hold/

## Section 4: Model Configuration

In [26]:
# Cell 9: Create Model Configuration

model_config = ModelConfig(
    input_size=n_features,
    hidden_size=256,
    num_layers=2,
    dropout=0.1,
    bidirectional=False,
    num_classes=2,
    input_seq_length=PREPROCESSOR_CONFIG["sequence_length"],
    classifier_hidden_size=32,
    # CNN-LSTM specific
    kernel_size=5,
    cnn_num_layers=3,
    cnn_dropout=0.1,
    lstm_dropout=0.1,
    classifier_dropout=0.1,
)

print("Model Configuration:")
print("=" * 40)
for field_name, value in asdict(model_config).items():
    print(f"  {field_name}: {value}")

Model Configuration:
  input_size: 177
  hidden_size: 256
  num_layers: 2
  dropout: 0.1
  bidirectional: False
  num_classes: 2
  input_seq_length: 36
  classifier_hidden_size: 32
  kernel_size: 5
  cnn_num_layers: 3
  cnn_dropout: 0.1
  lstm_dropout: 0.1
  classifier_dropout: 0.1


In [27]:
# Cell 10: Create Model

model = CNNLSTMSignalPredictor(model_config)
print(model)
print(f"\nTotal trainable parameters: {model.get_num_parameters():,}")

CNNLSTMSignalPredictor(
  input_size=177,
  projection_size=354,
  hidden_size=256,
  cnn_num_layers=3,
  kernel_size=5,
  cnn_dropout=0.1,
  lstm_num_layers=2,
  lstm_dropout=0.1,
  classifier_hidden_size=32,
  classifier_dropout=0.1,
  bidirectional=False,
  num_classes=2 (hold=0, trade=1),
  total_params=2,200,262
)

Total trainable parameters: 2,200,262


In [28]:
## Section 5: Training Configuration

In [29]:
# Cell 11: Helper Function for Loss Configuration

def configure_loss_settings(
    distributions: Dict[str, Dict],
    verbose: bool = True
) -> Dict:
    """
    Determine optimal loss function settings based on dataset distribution.
    
    Logic:
    - Extreme imbalance (ratio > 10): Use focal loss with high gamma, full class weights
    - High imbalance (ratio 5-10): Use focal loss with moderate gamma
    - Moderate imbalance (ratio 2-5): Use class weights only
    - Balanced (ratio < 2): Minimal adjustments
    
    Parameters
    ----------
    distributions : Dict[str, Dict]
        Distribution statistics from analyze_dataset_distribution
    verbose : bool
        Print reasoning
    
    Returns
    -------
    Dict
        Configuration dict with:
        - auto_class_weights: bool
        - class_weight_power: float (0.0-1.0)
        - focal_loss: bool
        - focal_gamma: float (0.0-5.0)
        - label_smoothing: float (0.0-0.2)
    """
    train_dist = distributions.get('train', {})
    imbalance_ratio = train_dist.get('imbalance_ratio', 1.0)
    trade_pct = train_dist.get('trade_pct', 50.0)
    
    if verbose:
        print("=" * 60)
        print("LOSS CONFIGURATION HELPER")
        print("=" * 60)
        print(f"Train imbalance ratio (hold/trade): {imbalance_ratio:.2f}")
        print(f"Trade percentage: {trade_pct:.2f}%")
    
    # Determine settings based on imbalance
    if imbalance_ratio > 10:
        config = {
            'auto_class_weights': True,
            'class_weight_power': 1.0,
            'focal_loss': True,
            'focal_gamma': 3.0,
            'label_smoothing': 0.1,
        }
        reason = "EXTREME imbalance (>10x) - Using focal loss with high gamma"
        
    elif imbalance_ratio > 5:
        config = {
            'auto_class_weights': True,
            'class_weight_power': 0.75,
            'focal_loss': True,
            'focal_gamma': 2.0,
            'label_smoothing': 0.1,
        }
        reason = "HIGH imbalance (5-10x) - Using focal loss with moderate settings"
        
    elif imbalance_ratio > 2:
        config = {
            'auto_class_weights': True,
            'class_weight_power': 0.5,
            'focal_loss': False,
            'focal_gamma': 2.0,
            'label_smoothing': 0.05,
        }
        reason = "MODERATE imbalance (2-5x) - Using class weights only"
        
    else:
        config = {
            'auto_class_weights': True,
            'class_weight_power': 0.25,
            'focal_loss': False,
            'focal_gamma': 2.0,
            'label_smoothing': 0.0,
        }
        reason = "BALANCED dataset (<2x) - Minimal adjustments"
    
    if verbose:
        print(f"\nDecision: {reason}")
        print(f"\nConfiguration:")
        for key, value in config.items():
            print(f"  {key}: {value}")
        print("=" * 60)
    
    return config


# Configure loss settings based on distribution
if dataset_distributions:
    loss_config = configure_loss_settings(dataset_distributions, verbose=True)
else:
    loss_config = {
        'auto_class_weights': True,
        'class_weight_power': 0.5,
        'focal_loss': False,
        'focal_gamma': 2.0,
        'label_smoothing': 0.05,
    }
    print("Using default loss configuration.")

LOSS CONFIGURATION HELPER
Train imbalance ratio (hold/trade): 0.74
Trade percentage: 57.46%

Decision: BALANCED dataset (<2x) - Minimal adjustments

Configuration:
  auto_class_weights: True
  class_weight_power: 0.25
  focal_loss: False
  focal_gamma: 2.0
  label_smoothing: 0.0


In [37]:
# Cell 12: Create Training Configuration

training_config = TrainingConfig(
    # Model architecture (for reference)
    hidden_size=model_config.hidden_size,
    num_layers=model_config.num_layers,
    dropout=model_config.dropout,
    
    # Training parameters
    epochs=400,
    batch_size=1024,
    learning_rate=5e-3,
    weight_decay=1e-3,
    optimizer='adamw',
    grad_clip_norm=1.0,
    
    # Learning rate scheduler
    scheduler='plateau',
    scheduler_patience=20,
    scheduler_factor=0.8,
    
    # Class imbalance handling (from helper function)
    auto_class_weights=False,
    class_weight_power=0.25,
    focal_loss=loss_config['focal_loss'],
    focal_gamma=loss_config['focal_gamma'],
    label_smoothing=0.05,
    
    # Data split (we provide pre-split datasets)
    val_split=0.2,
    test_split=0.2,
    
    # Early stopping
    early_stopping=False,
    patience=20,
    min_delta=1e-4,
    
    # Checkpointing
    checkpoint_dir=str(OUTPUT_DIR / "checkpoints"),
    save_best_only=True,
    
    # Device
    device='cuda',
    
    # Logging
    log_interval=50,
    verbose=False,
)

print("Training Configuration created.")
print(f"Epochs: {training_config.epochs}")
print(f"Batch size: {training_config.batch_size}")
print(f"Learning rate: {training_config.learning_rate}")
print(f"Focal loss: {training_config.focal_loss}")
print(f"Class weight power: {training_config.class_weight_power}")

Training Configuration created.
Epochs: 400
Batch size: 1024
Learning rate: 0.005
Focal loss: False
Class weight power: 0.25


In [38]:
## Section 6: MLflow Integration

In [39]:
# Cell 13: MLflow Setup and Logging Functions

def setup_mlflow(experiment_name: str) -> str:
    """
    Setup MLflow experiment.
    
    Parameters
    ----------
    experiment_name : str
        Name for the MLflow experiment
    
    Returns
    -------
    str
        Experiment ID
    """
    # Set tracking URI (default is local ./mlruns)
    mlflow.set_tracking_uri("http://localhost:5000")
    
    # Create or get experiment
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = mlflow.create_experiment(experiment_name)
    else:
        experiment_id = experiment.experiment_id
    
    mlflow.set_experiment(experiment_name)
    
    print(f"MLflow experiment: {experiment_name}")
    print(f"Experiment ID: {experiment_id}")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    
    return experiment_id


def log_training_params(
    model_config: ModelConfig,
    training_config: TrainingConfig,
    preprocessor_config: Dict,
    symbols: List[str],
    distributions: Dict[str, Dict]
):
    """
    Log all configuration parameters to MLflow.
    """
    # Model params
    mlflow.log_params({
        "model.input_size": model_config.input_size,
        "model.hidden_size": model_config.hidden_size,
        "model.num_layers": model_config.num_layers,
        "model.dropout": model_config.dropout,
        "model.bidirectional": model_config.bidirectional,
        "model.classifier_hidden_size": model_config.classifier_hidden_size,
        "model.kernel_size": model_config.kernel_size,
        "model.cnn_num_layers": model_config.cnn_num_layers,
        "model.cnn_dropout": model_config.cnn_dropout,
        "model.lstm_dropout": model_config.lstm_dropout,
    })
    
    # Training params
    mlflow.log_params({
        "train.epochs": training_config.epochs,
        "train.batch_size": training_config.batch_size,
        "train.learning_rate": training_config.learning_rate,
        "train.weight_decay": training_config.weight_decay,
        "train.optimizer": training_config.optimizer,
        "train.scheduler": training_config.scheduler,
        "train.focal_loss": training_config.focal_loss,
        "train.focal_gamma": training_config.focal_gamma,
        "train.class_weight_power": training_config.class_weight_power,
        "train.label_smoothing": training_config.label_smoothing,
        "train.early_stopping": training_config.early_stopping,
        "train.patience": training_config.patience,
    })
    
    # Preprocessor params
    mlflow.log_params({
        "data.sequence_length": preprocessor_config["sequence_length"],
        "data.target_shift": preprocessor_config["target_shift"],
        "data.stride": preprocessor_config["stride"],
        "data.remove_raw_indicators": preprocessor_config["remove_raw_indicators"],
        "data.num_coins": len(symbols),
    })
    
    # Distribution info
    train_dist = distributions.get('train', {})
    mlflow.log_params({
        "dist.train_samples": train_dist.get('total', 0),
        "dist.train_trade_pct": round(train_dist.get('trade_pct', 0), 2),
        "dist.imbalance_ratio": round(train_dist.get('imbalance_ratio', 1), 2),
    })


def log_training_metrics(history: TrainingHistory, epoch: int):
    """
    Log training metrics for a single epoch.
    """
    mlflow.log_metrics({
        "train_loss": history.train_losses[-1],
        "val_loss": history.val_losses[-1],
        "train_accuracy": history.train_accuracies[-1],
        "val_accuracy": history.val_accuracies[-1],
        "learning_rate": history.learning_rates[-1],
    }, step=epoch)


def log_evaluation_results(results: Dict[str, Dict]):
    """
    Log final evaluation metrics.
    """
    for split_name, metrics in results.items():
        if metrics is None:
            continue
        prefix = f"{split_name}_"
        mlflow.log_metrics({
            f"{prefix}accuracy": metrics['accuracy'],
            f"{prefix}precision": metrics['precision'],
            f"{prefix}recall": metrics['recall'],
            f"{prefix}f1": metrics['f1'],
            f"{prefix}hold_f1": metrics['hold_f1'],
            f"{prefix}trade_f1": metrics['trade_f1'],
        })


def log_model_artifact(model: torch.nn.Module, preprocessor: VectorBTDataPreprocessor, output_dir: Path):
    """
    Log model and preprocessor as MLflow artifacts.
    """
    # Log PyTorch model
    mlflow.pytorch.log_model(model, "model")
    
    # Log preprocessor
    preprocessor_path = output_dir / "preprocessor_mlflow.pkl"
    preprocessor.save(preprocessor_path)
    mlflow.log_artifact(str(preprocessor_path))
    preprocessor_path.unlink()  # Clean up temp file

# Cell 14: Setup MLflow experiment

In [40]:
## Section 7: Training Loop

In [41]:
experiment_id = setup_mlflow(MLFLOW_EXPERIMENT)

MLflow experiment: multi_coin_lstm_training
Experiment ID: 1
Tracking URI: http://localhost:5000


In [42]:
# Cell 15: Training Function with MLflow

def train_with_mlflow(
    model: torch.nn.Module,
    training_config: TrainingConfig,
    train_dataset: SignalDataset,
    val_dataset: SignalDataset,
    test_dataset: SignalDataset,
    model_config: ModelConfig,
    preprocessor_config: Dict,
    symbols: List[str],
    distributions: Dict[str, Dict],
    preprocessor: VectorBTDataPreprocessor,
    output_dir: Path
) -> Tuple[TrainingHistory, Dict, Trainer, str]:
    """
    Train model with full MLflow tracking.
    
    Returns
    -------
    Tuple[TrainingHistory, Dict, Trainer, str]
        (history, evaluation_results, trainer, run_id)
    """
    with mlflow.start_run() as run:
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}")
        print("=" * 60)
        
        # Log all parameters
        log_training_params(
            model_config, training_config, 
            preprocessor_config,
            symbols, distributions
        )
        
        # Create trainer (use preprocessor for consistency, though VectorBTDataPreprocessor
        # doesn't have the same interface as DataPreprocessor - we'll pass None)
        trainer = Trainer(
            model=model,
            config=training_config,
            preprocessor=None  # VectorBTDataPreprocessor has different interface
        )
        
        # Store test dataset for evaluation
        trainer.test_dataset = test_dataset
        
        # Define callback for epoch-level logging
        def mlflow_callback(epoch: int, history: TrainingHistory):
            log_training_metrics(history, epoch)
        
        # Train
        print("\nStarting training...")
        history = trainer.train(
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            callbacks=[mlflow_callback]
        )
        
        # Evaluate
        print("\nEvaluating model...")
        results = trainer.evaluate_all(verbose=True)
        
        # Log evaluation metrics
        log_evaluation_results(results)
        
        # Log final metrics
        mlflow.log_metrics({
            "best_epoch": history.best_epoch + 1,
            "best_val_loss": history.best_val_loss,
            "epochs_trained": len(history.train_losses),
        })
        
        # Log model artifact
        log_model_artifact(model, preprocessor, output_dir)
        
        print(f"\nMLflow Run completed: {run_id}")
        
    return history, results, trainer, run_id

In [43]:
# Cell 16: Execute Training

if multi_coin_datasets:
    # Get symbols list
    symbols = list(coin_dataframes.keys())
    
    # Execute training with MLflow
    history, eval_results, trainer, run_id = train_with_mlflow(
        model=model,
        training_config=training_config,
        train_dataset=multi_coin_datasets['train'],
        val_dataset=multi_coin_datasets['val'],
        test_dataset=multi_coin_datasets['test'],
        model_config=model_config,
        preprocessor_config=PREPROCESSOR_CONFIG,
        symbols=symbols,
        distributions=dataset_distributions,
        preprocessor=preprocessor,
        output_dir=OUTPUT_DIR
    )
    
    print(f"\nTraining complete!")
    print(f"Best epoch: {history.best_epoch + 1}")
    print(f"Best validation loss: {history.best_val_loss:.4f}")
    print(f"MLflow Run ID: {run_id}")
else:
    print("No datasets available for training.")

MLflow Run ID: defd87771d4b412bb603c9f4c114302d

Starting training...

Evaluating model...


2026/01/20 21:43:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



COMPREHENSIVE MODEL EVALUATION REPORT (Binary: hold=0, trade=1)

--------------------------------------------------------------------------------
OVERALL METRICS COMPARISON
--------------------------------------------------------------------------------
Metric          TRAIN                VAL                  
--------------------------------------------------------------------------------
Accuracy        0.5312               0.4702               
Precision       0.4612               0.4572               
Recall          0.5312               0.4702               
F1              0.4521               0.3321               

--------------------------------------------------------------------------------
TRAIN DATASET - Per-Class Metrics
--------------------------------------------------------------------------------
Class      Precision    Recall       F1           Support   
------------------------------------------------------------
hold       0.3272       0.0964       0.1489       

2026/01/20 21:43:10 WARNING mlflow.utils.requirements_utils: Found torch version (2.9.1+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.9.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/20 21:43:19 WARNING mlflow.utils.requirements_utils: Found torch version (2.9.1+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.9.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.



MLflow Run completed: defd87771d4b412bb603c9f4c114302d
🏃 View run thundering-bird-44 at: http://localhost:5000/#/experiments/1/runs/defd87771d4b412bb603c9f4c114302d
🧪 View experiment at: http://localhost:5000/#/experiments/1

Training complete!
Best epoch: 3
Best validation loss: 0.7280
MLflow Run ID: defd87771d4b412bb603c9f4c114302d


## Section 8: Model Configuration (Already defined above)

# Note: Model configuration was defined in Section 4
# This cell is kept for notebook structure reference

In [ ]:
# This cell previously contained model config - now handled in Section 4

# Skip this cell - model config is defined earlier

In [ ]:
# Skip - model creation handled in Section 4

# Skip - training config helper handled in Section 5

In [ ]:
# Skip - loss config handled in Section 5

In [ ]:
# Skip - training config creation handled in Section 5

In [ ]:
# Skip - training config done in Section 5

# Skip - MLflow section handled in Section 6

In [ ]:
# Skip - MLflow setup handled in Section 6

In [ ]:
# Skip - MLflow experiment setup handled in Section 6

# Skip - training loop section handled in Section 7

In [ ]:
# Skip - training function handled in Section 7

In [ ]:
# Skip - training execution handled in Section 7

In [ ]:
# Utility cell - uncomment if needed
# !git pull

## Section 8: Evaluation

In [ ]:
# Cell 17: Detailed Evaluation Report

# Print comprehensive evaluation report
if 'trainer' in dir() and trainer is not None:
    trainer.print_evaluation_report(eval_results)
else:
    print("Training not completed yet.")

## Section 9: Visualization and Plots

In [ ]:
# Cell 18: Training Curves Plot

def plot_training_curves(
    history: TrainingHistory,
    output_path: Optional[Path] = None
) -> None:
    """
    Create and optionally save training curves visualization.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    epochs = range(1, len(history.train_losses) + 1)
    
    # Loss curves
    ax = axes[0, 0]
    ax.plot(epochs, history.train_losses, 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, history.val_losses, 'r-', label='Val Loss', linewidth=2)
    ax.axvline(x=history.best_epoch + 1, color='g', linestyle='--', 
               label=f'Best Epoch ({history.best_epoch + 1})', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax = axes[0, 1]
    ax.plot(epochs, history.train_accuracies, 'b-', label='Train Acc', linewidth=2)
    ax.plot(epochs, history.val_accuracies, 'r-', label='Val Acc', linewidth=2)
    ax.axvline(x=history.best_epoch + 1, color='g', linestyle='--', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Training & Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Learning rate
    ax = axes[1, 0]
    ax.plot(epochs, history.learning_rates, 'purple', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    # Loss difference (overfitting indicator)
    ax = axes[1, 1]
    loss_diff = np.array(history.val_losses) - np.array(history.train_losses)
    ax.plot(epochs, loss_diff, 'orange', linewidth=2)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax.fill_between(epochs, loss_diff, 0, where=(loss_diff > 0), 
                    alpha=0.3, color='red', label='Overfitting')
    ax.fill_between(epochs, loss_diff, 0, where=(loss_diff <= 0), 
                    alpha=0.3, color='green', label='Underfitting')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Loss - Train Loss')
    ax.set_title('Overfitting Indicator')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_path}")
    
    plt.show()

# Generate training curves
if 'history' in dir() and history is not None:
    plot_training_curves(history, OUTPUT_DIR / "training_curves.png")
else:
    print("Training not completed yet.")

In [ ]:
# Cell 19: Confusion Matrices

def plot_confusion_matrices(
    results: Dict[str, Dict],
    output_path: Optional[Path] = None
) -> None:
    """
    Plot confusion matrices for all dataset splits.
    """
    splits = [k for k, v in results.items() if v is not None]
    n_splits = len(splits)
    
    if n_splits == 0:
        print("No results to plot.")
        return
    
    fig, axes = plt.subplots(1, n_splits, figsize=(5 * n_splits, 4))
    if n_splits == 1:
        axes = [axes]
    
    for idx, split_name in enumerate(splits):
        metrics = results[split_name]
        cm = np.array(metrics['confusion_matrix'])
        
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Hold', 'Trade'],
            yticklabels=['Hold', 'Trade'],
            ax=axes[idx]
        )
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')
        axes[idx].set_title(f'{split_name.upper()}\nF1: {metrics["f1"]:.3f}, '
                           f'Trade F1: {metrics["trade_f1"]:.3f}')
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_path}")
    
    plt.show()

# Plot confusion matrices
if 'eval_results' in dir() and eval_results:
    plot_confusion_matrices(eval_results, OUTPUT_DIR / "confusion_matrices.png")
else:
    print("Evaluation results not available.")

In [ ]:
# Cell 20: Per-Coin Performance Analysis

def analyze_per_coin_performance(
    model: torch.nn.Module,
    split_data: Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]],
    device: torch.device
) -> pd.DataFrame:
    """
    Analyze model performance on each coin separately (test set only).
    """
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    
    model.eval()
    results = []
    
    for name, splits in split_data.items():
        X_test, y_test = splits['test']
        
        if len(X_test) == 0:
            continue
        
        # Create dataset and get predictions
        dataset = SignalDataset(X_test, y_test, device=device)
        
        with torch.no_grad():
            features = dataset.features
            logits = model(features)
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            targets = dataset.targets.cpu().numpy()
        
        # Calculate metrics
        results.append({
            'Name': name,
            'Test Samples': len(targets),
            'Accuracy': accuracy_score(targets, preds),
            'F1': f1_score(targets, preds, average='weighted', zero_division=0),
            'Trade F1': f1_score(targets, preds, pos_label=1, zero_division=0),
            'Trade Precision': precision_score(targets, preds, pos_label=1, zero_division=0),
            'Trade Recall': recall_score(targets, preds, pos_label=1, zero_division=0),
            'Trade %': (targets == 1).mean() * 100
        })
    
    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values('Trade F1', ascending=False)
    
    print("=" * 100)
    print("PER-COIN TEST PERFORMANCE")
    print("=" * 100)
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else x))
    print("=" * 100)
    
    return df

# Analyze per-coin performance
if 'model' in dir() and 'split_coin_data' in dir() and split_coin_data:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    per_coin_results = analyze_per_coin_performance(model, split_coin_data, device)
else:
    print("Model or split data not available.")
    per_coin_results = pd.DataFrame()

In [ ]:
# Cell 21: Per-Coin Performance Visualization

def plot_per_coin_performance(
    per_coin_df: pd.DataFrame,
    output_path: Optional[Path] = None
) -> None:
    """
    Visualize per-coin performance metrics.
    """
    if per_coin_df.empty:
        print("No per-coin results to plot.")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Sort by Trade F1
    df_sorted = per_coin_df.sort_values('Trade F1', ascending=True)
    
    # Trade F1 by coin
    ax = axes[0]
    colors = plt.cm.RdYlGn(df_sorted['Trade F1'].values)
    ax.barh(df_sorted['Name'], df_sorted['Trade F1'], color=colors)
    ax.set_xlabel('Trade F1 Score')
    ax.set_title('Trade F1 Score by Coin')
    ax.axvline(x=df_sorted['Trade F1'].mean(), color='red', linestyle='--', 
               label=f'Mean: {df_sorted["Trade F1"].mean():.3f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Accuracy vs Trade F1 scatter
    ax = axes[1]
    scatter = ax.scatter(
        per_coin_df['Accuracy'], 
        per_coin_df['Trade F1'],
        c=per_coin_df['Trade %'],
        cmap='viridis',
        s=100,
        alpha=0.7
    )
    for idx, row in per_coin_df.iterrows():
        ax.annotate(row['Name'], (row['Accuracy'], row['Trade F1']), 
                   fontsize=8, ha='center', va='bottom')
    ax.set_xlabel('Accuracy')
    ax.set_ylabel('Trade F1')
    ax.set_title('Accuracy vs Trade F1 (color = Trade %)')
    plt.colorbar(scatter, ax=ax, label='Trade %')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_path}")
    
    plt.show()

# Plot per-coin performance
if 'per_coin_results' in dir() and not per_coin_results.empty:
    plot_per_coin_performance(per_coin_results, OUTPUT_DIR / "per_coin_performance.png")
else:
    print("Per-coin results not available.")

## Section 10: Summary Statistics

In [ ]:
# Cell 22: Experiment Summary Dataclass

@dataclass
class ExperimentSummary:
    """Summary of experiment for comparison."""
    timestamp: str
    run_id: str
    
    # Data info
    num_coins: int
    total_train_samples: int
    total_val_samples: int
    total_test_samples: int
    train_trade_pct: float
    imbalance_ratio: float
    
    # Model info
    model_type: str
    num_parameters: int
    hidden_size: int
    num_layers: int
    cnn_num_layers: int
    
    # Training info
    epochs_trained: int
    best_epoch: int
    best_val_loss: float
    learning_rate: float
    focal_loss: bool
    focal_gamma: float
    class_weight_power: float
    label_smoothing: float
    
    # Results
    train_accuracy: float
    val_accuracy: float
    test_accuracy: float
    train_f1: float
    val_f1: float
    test_f1: float
    test_trade_f1: float
    test_trade_precision: float
    test_trade_recall: float
    
    def to_dict(self) -> Dict:
        return asdict(self)


def create_experiment_summary(
    history: TrainingHistory,
    eval_results: Dict[str, Dict],
    model: torch.nn.Module,
    datasets: Dict[str, SignalDataset],
    training_config: TrainingConfig,
    model_config: ModelConfig,
    distributions: Dict[str, Dict],
    symbols: List[str],
    run_id: str
) -> ExperimentSummary:
    """
    Create experiment summary for tracking and comparison.
    """
    train_dist = distributions.get('train', {})
    
    summary = ExperimentSummary(
        timestamp=datetime.now().isoformat(),
        run_id=run_id,
        
        num_coins=len(symbols),
        total_train_samples=len(datasets.get('train', [])),
        total_val_samples=len(datasets.get('val', [])),
        total_test_samples=len(datasets.get('test', [])),
        train_trade_pct=round(train_dist.get('trade_pct', 0), 2),
        imbalance_ratio=round(train_dist.get('imbalance_ratio', 1), 2),
        
        model_type=model.__class__.__name__,
        num_parameters=model.get_num_parameters(),
        hidden_size=model_config.hidden_size,
        num_layers=model_config.num_layers,
        cnn_num_layers=model_config.cnn_num_layers,
        
        epochs_trained=len(history.train_losses),
        best_epoch=history.best_epoch + 1,
        best_val_loss=round(history.best_val_loss, 4),
        learning_rate=training_config.learning_rate,
        focal_loss=training_config.focal_loss,
        focal_gamma=training_config.focal_gamma,
        class_weight_power=training_config.class_weight_power,
        label_smoothing=training_config.label_smoothing,
        
        train_accuracy=round(eval_results.get('train', {}).get('accuracy', 0), 4),
        val_accuracy=round(eval_results.get('val', {}).get('accuracy', 0), 4),
        test_accuracy=round(eval_results.get('test', {}).get('accuracy', 0), 4),
        train_f1=round(eval_results.get('train', {}).get('f1', 0), 4),
        val_f1=round(eval_results.get('val', {}).get('f1', 0), 4),
        test_f1=round(eval_results.get('test', {}).get('f1', 0), 4),
        test_trade_f1=round(eval_results.get('test', {}).get('trade_f1', 0), 4),
        test_trade_precision=round(eval_results.get('test', {}).get('trade_precision', 0), 4),
        test_trade_recall=round(eval_results.get('test', {}).get('trade_recall', 0), 4),
    )
    
    return summary

In [ ]:
# Cell 23: Create and Display Experiment Summary

if all(var in dir() for var in ['history', 'eval_results', 'model', 'multi_coin_datasets', 
                                  'training_config', 'model_config', 'dataset_distributions', 'run_id']):
    symbols = list(coin_dataframes.keys())
    
    summary = create_experiment_summary(
        history, eval_results, model, multi_coin_datasets,
        training_config, model_config, dataset_distributions,
        symbols, run_id
    )
    
    print("\n" + "=" * 70)
    print("EXPERIMENT SUMMARY")
    print("=" * 70)
    
    summary_dict = summary.to_dict()
    sections = {
        'General': ['timestamp', 'run_id'],
        'Data': ['num_coins', 'total_train_samples', 'total_val_samples', 'total_test_samples', 
                 'train_trade_pct', 'imbalance_ratio'],
        'Model': ['model_type', 'num_parameters', 'hidden_size', 'num_layers', 'cnn_num_layers'],
        'Training': ['epochs_trained', 'best_epoch', 'best_val_loss', 'learning_rate',
                     'focal_loss', 'focal_gamma', 'class_weight_power', 'label_smoothing'],
        'Results': ['train_accuracy', 'val_accuracy', 'test_accuracy', 'train_f1', 'val_f1',
                    'test_f1', 'test_trade_f1', 'test_trade_precision', 'test_trade_recall']
    }
    
    for section_name, keys in sections.items():
        print(f"\n{section_name}:")
        for key in keys:
            value = summary_dict[key]
            if isinstance(value, float) and not isinstance(value, bool):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")
    
    print("\n" + "=" * 70)
else:
    print("Training not completed - cannot create summary.")

In [ ]:
# Cell 24: Save Experiment Summary

def save_experiment_summary(
    summary: ExperimentSummary,
    output_dir: Path
) -> Path:
    """
    Save experiment summary to JSON.
    """
    timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    summary_path = output_dir / f"experiment_summary_{timestamp_str}.json"
    
    with open(summary_path, 'w') as f:
        json.dump(summary.to_dict(), f, indent=2)
    
    print(f"Saved summary: {summary_path}")
    return summary_path

# Save the summary
if 'summary' in dir():
    summary_path = save_experiment_summary(summary, OUTPUT_DIR)
else:
    print("Summary not available.")

In [ ]:
# Cell 25: Per-Coin Results to CSV

# Save per-coin results
if 'per_coin_results' in dir() and not per_coin_results.empty:
    per_coin_path = OUTPUT_DIR / "per_coin_test_results.csv"
    per_coin_results.to_csv(per_coin_path, index=False)
    print(f"Saved per-coin results: {per_coin_path}")
else:
    print("Per-coin results not available.")

## Next Steps

1. **Generate CSV Data**: Run `vectorbt_optimizer.optimize_all()` to create CSV files in `notebooks/csv/`
2. **View MLflow UI**: Run `mlflow ui` in terminal to view experiment tracking
3. **Compare Experiments**: Modify configurations and re-run to compare results
4. **Hyperparameter Tuning**: Adjust model and training configs based on results
5. **Production**: Use best model for trading signal predictions

In [ ]:
# Cell 26: Final Summary - Files Created

print("\n" + "=" * 70)
print("FILES CREATED")
print("=" * 70)

# List all created files
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f"  {path.relative_to(OUTPUT_DIR)}: {size_kb:.1f} KB")

print("\n" + "=" * 70)
print("To view MLflow experiments, run in terminal:")
print("  cd notebooks && mlflow ui")
print("Then open http://localhost:5000 in your browser")
print("=" * 70)

# Save preprocessor for inference
if 'preprocessor' in dir() and preprocessor._is_fitted:
    preprocessor_path = OUTPUT_DIR / "vectorbt_preprocessor.pkl"
    preprocessor.save(preprocessor_path)
    print(f"\nSaved VectorBTDataPreprocessor: {preprocessor_path}")